# MNIST MLP3: AdamW baseline versus trace-log RG component removal

This notebook trains two identically initialized `784 -> 512 -> 512 -> 10` ReLU MLPs on the same MNIST minibatches:

1. AdamW baseline.
2. AdamW plus the trace-log RG wrapper.

The default RG mode is the conservative one-sided rule: it removes only first-order contraction of the working ECS trace-log volume. The working ECS rank is refreshed from the midpoint between WeightWatcher's PL and ERG boundaries at every epoch. The layer is never truncated or projected onto that ECS.

At every checkpoint, the notebook tracks WeightWatcher `alpha`, `detX_num`, `num_pl_spikes`, `ERG_gap`, the midpoint trace-log residual, and the new logarithmic-shell `beta_E` term.

In [ ]:
from pathlib import Path
import subprocess
import sys

PACKAGE_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "rg_trace_log").is_dir():
        PACKAGE_ROOT = candidate
        break
    nested = candidate / "optimizers" / "trace_log_tracker"
    if (nested / "rg_trace_log").is_dir():
        PACKAGE_ROOT = nested
        break
if PACKAGE_ROOT is None:
    raise RuntimeError("Open this notebook from a clone of the rg_optimizers repository.")

INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(PACKAGE_ROOT / "requirements.txt"),
    ])
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))
print("Package root:", PACKAGE_ROOT)

In [ ]:
import pandas as pd
from IPython.display import display

import torch
import weightwatcher as ww

from rg_trace_log import MNISTExperimentConfig, run_mnist_comparison
from rg_trace_log.plotting import (
    plot_accuracy,
    plot_beta,
    plot_correction_summary,
    plot_weightwatcher_metric,
)

print("Torch:", torch.__version__)
print("WeightWatcher:", getattr(ww, "__version__", "unknown"))

## Configure the experiment

In [ ]:
config = MNISTExperimentConfig(
    seed=1337,
    epochs=20,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-2,
    grad_clip_norm=1.0,

    # "one_sided" protects against contraction toward the trivial branch.
    # Other available tests are "tangent" and "tracking".
    rg_mode="one_sided",
    rg_normalization="weightwatcher",
    rg_gamma=0.10,
    rg_ridge_relative=1e-6,
    rg_min_retained=5,
    rg_correction_scale=1.0,
    rg_max_correction_ratio=0.10,
    rg_apply_every_steps=25,
    rg_warmup_steps=0,

    ww_min_evals=10,
    ww_max_evals=None,
    n_log_shells=5,
    min_retained_for_beta=20,
    min_decades_for_beta=0.50,
)
config

## Run AdamW baseline and AdamW + TraceLogRG

In [ ]:
result = run_mnist_comparison(
    config,
    data_dir=PACKAGE_ROOT / "data",
    progress=True,
)
result.save(PACKAGE_ROOT / "results")
print("Results saved to", (PACKAGE_ROOT / "results").resolve())

## Core result tables

In [ ]:
display(result.performance.tail(8))

columns = [
    "run", "epoch", "layer_name", "alpha", "detX_num", "num_pl_spikes",
    "ERG_gap", "m_midpoint", "trace_log_midpoint_per_eval",
    "beta_E_midpoint", "shell_energy_rms_midpoint",
    "rg_shell_shift_rms_midpoint", "scale_balance_reliable",
]
display(result.weightwatcher[[c for c in columns if c in result.weightwatcher.columns]].tail(18))
display(result.correction_summary.tail(12))

## Accuracy

In [ ]:
plot_accuracy(result.performance)

## WeightWatcher alpha

In [ ]:
plot_weightwatcher_metric(
    result.weightwatcher,
    "alpha",
    "WeightWatcher alpha by optimizer and layer",
    "WeightWatcher alpha",
    reference=2.0,
)

## WeightWatcher ERG gap

In [ ]:
plot_weightwatcher_metric(
    result.weightwatcher,
    "ERG_gap",
    "WeightWatcher ERG gap by optimizer and layer",
    "detX_num - num_pl_spikes",
    reference=0.0,
)

## New logarithmic-shell beta term

In [ ]:
plot_beta(result.weightwatcher)

## RG correction size

In [ ]:
plot_correction_summary(result.correction_summary)

## Interpretation checklist

- Did the RG run preserve baseline train and test accuracy?
- Did `fc1` spend less time below `alpha = 2`?
- Did `fc3` move down from the weak-tail `alpha > 2` regime?
- Did the ERG gaps remain closer to zero?
- Did reliable midpoint `beta_E` values move closer to zero without increasing the full shell-energy RMS?
- Were the RG corrections small relative to the completed AdamW steps?

Change `rg_mode` to `"tangent"` to remove all first-order trace-log drift, or to `"tracking"` to actively contract the current residual.